# Genebass single-variant BETAs → UKBBGym APPV proxy

We extract single-variant SAIGE **BETAs** from the Genebass Hail MatrixTable (`$GENEBASS_MT_PATH`, ~1 TB) to use as a proxy for **APPV** (average phenotype per variant carrier) in the UKBBGym benchmark — avoiding a return to raw UKB genotypes.

**Input:** the UKBBGym association file — 2,289 `(gene, phenotype)` pairs (699 genes × 121 phenotypes) chosen via pLoF burden tests.

**Output:** one parquet (`genebass_betas/genebass_betas_127phenos_allvars.parquet`) with one row per `(variant, phenotype)`, restricted to **exactly those 2,289 pairs**, with **all variants** in each gene (any annotation):

| `id` | `region` | `phenotype` | `phenocode` | `mean_pheno_value` | `SE` | `Pvalue` | `AF` | `n_cases` |
|---|---|---|---|---|---|---|---|---|
| `chr:pos:ref:alt` | Ensembl ID | `_int` name | UKB code | SAIGE BETA | | | | |

See `README.md` for full details. The MT is never loaded into memory — we prune by `locus` interval and collect per phenotype.

In [ ]:
import polars as pl
import hail as hl

hl.init()
hl.default_reference('GRCh38')

## 0. Map Ensembl IDs → gene symbols (biomart, sanity only)

The association file uses Ensembl IDs (`region`), but the MT's `gene` field holds gene **symbols**. This biomart mapping confirms all 699 IDs resolve to a symbol, but biomart uses *current* symbols. The mapping actually used to filter the MT comes from **GENCODE v29** in step 5 (`region_to_symbol`), because the MT was annotated with that version — using current symbols would silently drop the 7 genes renamed since 2018.

**Input:** `genes_info_biomart.parquet` + `ensembl_ids`. **Output:** `ensembl_to_symbol` (current symbols, for reference).

In [ ]:
BIOMART_PATH = 'PATH_TO_FILE'

gene_map = (
    pl.read_parquet(BIOMART_PATH)
    .select(['gene_stable_id', 'gene_name'])
    .unique('gene_stable_id')
)

ensembl_to_symbol = dict(zip(gene_map['gene_stable_id'].to_list(), gene_map['gene_name'].to_list()))

---
# Convert hail matrix table to appv parquet

## 1. Load the Genebass MatrixTable

**Input:** the MT on disk (`MT_PATH`). **Output:** a Hail `MatrixTable` handle `mt` — 8,074,878 variants (rows, keyed by `locus, alleles`) × 4,529 phenotypes (cols, keyed by `trait_type, phenocode, pheno_sex, coding, modifier`), with per-(variant,phenotype) entry fields `BETA, SE, Pvalue, AF, ...`. Nothing is read into memory yet; the cells below just inspect the schema and a few rows/cols.

In [ ]:
MT_PATH = os.environ["GENEBASS_MT_PATH"]
OUT_PARQUET = 'PATH_TO_FILE'

mt = hl.read_matrix_table(MT_PATH)
mt.describe()

In [ ]:
# Quick look at dimensions and a few rows
print(f"Rows (variants): {mt.count_rows()}")
print(f"Cols (phenotypes): {mt.count_cols()}")

# Flatten nested call_stats struct before show() — Hail bug with nested structs in show()
rows = mt.rows()
rows.select(
    'markerID', 'gene', 'annotation',
    AC=rows.call_stats.AC,
    AF=rows.call_stats.AF,
    AN=rows.call_stats.AN,
    homozygote_count=rows.call_stats.homozygote_count,
).show(5)

In [ ]:
# Show column (phenotype) metadata
mt.cols().show(5)

## 2. Load the UKBBGym association pairs

**Input:** `regenie_127phenotypes_mac20_lofteeHC_EUR_correlations.parquet`. **Output:** Polars DataFrame `assoc` — 2,289 rows, columns `region` (Ensembl gene ID), `phenotype` (`_int` name), `annotation` (`loftee_hc`), `n_variants`, `correlation`; plus the unique lists `ensembl_ids` (699) and `pheno_names` (121). These define which gene–phenotype pairs we extract.

In [ ]:
import polars as pl
import re

ASSOC_PATH = 'PATH_TO_FILE'

assoc = pl.read_parquet(ASSOC_PATH)
ensembl_ids = assoc['region'].unique().to_list()
pheno_names = assoc['phenotype'].unique().to_list()
print(f"{len(ensembl_ids)} unique genes (Ensembl IDs), {len(pheno_names)} unique phenotypes")
assoc

## 3. Match phenotypes to Genebass phenocodes

The MT identifies phenotypes by `phenocode` (+ trait_type, etc.), the association file by human-readable `_int` names. We match them on a normalized form of the MT `description` field (lowercased, all non-alphanumerics stripped).

**Input:** MT column metadata read directly from `MT_PATH/cols` (4,529 rows — fast, no MT scan) + `pheno_names`. **Output:** `pheno_map` — one row per matched phenotype (119/121) with `phenotype, phenocode, trait_type, pheno_sex, coding, modifier, n_cases`. The 2 unmatched (`weight_impedance_int`, `body_mass_index_bmi_impedance_int`) have no Genebass counterpart and are dropped.

In [ ]:
# Map Ensembl IDs to gene symbols, and report any unmapped IDs.
gene_symbols = list({ensembl_to_symbol[e] for e in ensembl_ids if e in ensembl_to_symbol})
unmapped_ensembl = [e for e in ensembl_ids if e not in ensembl_to_symbol]

print(f"Mapped {len(gene_symbols)}/{len(ensembl_ids)} Ensembl IDs to gene symbols")
if unmapped_ensembl:
    print(f"Unmapped: {unmapped_ensembl}")

col_meta = (
    hl.read_table(MT_PATH + '/cols')
    .select('trait_type', 'phenocode', 'pheno_sex', 'coding', 'modifier', 'n_cases', 'description')
    .to_pandas()
)

def normalize(s):
    s = str(s).lower()
    s = re.sub(r'_int$', '', s)       # strip trailing _int
    s = re.sub(r'[^a-z0-9]', '', s)  # drop everything except letters/digits
    return s                           # "fat-free" and "fatfree" both → "fatfree"

# Build lookup: normalized form → original phenotype name (with _int)
pheno_name_norm = {normalize(p): p for p in pheno_names}
pheno_name_norm

In [ ]:
import pandas as pd

mapping_rows = []
for _, row in col_meta.iterrows():
    key = normalize(row['description'])
    if key in pheno_name_norm:
        mapping_rows.append({
            'phenotype': pheno_name_norm[key],  # already has _int suffix
            'phenocode': row['phenocode'],
            'trait_type': row['trait_type'],
            'pheno_sex': row['pheno_sex'],
            'coding': row['coding'],
            'modifier': row['modifier'],
            'n_cases': row['n_cases'],
        })

pheno_map = pd.DataFrame(mapping_rows).drop_duplicates('phenotype')
matched = set(pheno_map['phenotype'])
unmatched = [p for p in pheno_names if p not in matched]

print(f"Matched {len(pheno_map)}/{len(pheno_names)} phenotypes")
if unmatched:
    print(f"Unmatched phenotypes: {unmatched}")
pheno_map

## 4. Gene coordinates **and symbols** from GENCODE v29

Genebass annotated variants with **VEP v95**, which ships the **GENCODE v29** gene set (Karczewski et al. 2022, *Cell Genomics*). So the MT's `gene` field carries v29 symbols and v29 boundaries — we must use v29, not a newer release, or we'd silently miss genes/variants (measured on our 699 genes: 7 symbol changes, ~20 genes with >5 kb boundary drift vs v39).

We use v29 both to (a) build `locus` intervals for cheap partition pruning and (b) get the `region → symbol` map that matches the MT's `gene` field.

**Input:** `gencode.v29.annotation.gtf.gz` + `ensembl_ids`. **Output:** Polars `gene_coords` (697/699 genes; 2 don't exist in v29) with `region, contig, start, end, gene_name`, and the dict `region_to_symbol`.

In [ ]:
import gzip, re

# Gene coordinates AND symbols from GENCODE v29 — the version Genebass actually used.
# Genebass annotated variants with VEP v95 (Karczewski et al. 2022, Cell Genomics),
# and Ensembl/VEP release 95 ships the GENCODE v29 gene set. So the MT's `gene`
# field holds v29 symbols and v29 boundaries. Using v39 here would (measured on our
# 699 genes): (a) silently drop 7 genes whose symbol changed since 2018 (e.g. the MT
# has v29 'GARS'/'VARS', v39 calls them 'GARS1'/'VARS1') and (b) miss variants for
# ~20 genes whose boundaries drifted >5 kb. So we use v29 for both.
GTF = 'PATH_TO_FILE'
want = set(ensembl_ids)                       # 699 Ensembl IDs (version-stripped)
gid_re = re.compile(r'gene_id "([^".]+)')     # capture stops at '.' -> drops version
gname_re = re.compile(r'gene_name "([^"]+)"')

coords = {}  # ensembl_id -> dict(contig, start, end, gene_name)
with gzip.open(GTF, 'rt') as fh:
    for line in fh:
        if line[0] == '#':
            continue
        f = line.split('\t', 8)
        if f[2] != 'gene':
            continue
        m = gid_re.search(f[8])
        if not m or m.group(1) not in want:
            continue
        sym = gname_re.search(f[8])
        coords[m.group(1)] = {
            'region': m.group(1), 'contig': f[0],
            'start': int(f[3]), 'end': int(f[4]),
            'gene_name': sym.group(1) if sym else None,  # v29 symbol == the MT's `gene` field
        }

gene_coords = pl.DataFrame(list(coords.values()))
missing = want - set(coords)
print(f"GENCODE v29 coordinates for {gene_coords.height}/{len(want)} genes; missing: {len(missing)}")
if missing:
    print("  (absent in GENCODE v29 -> no Genebass data under these IDs):", sorted(missing))

# Authoritative region -> symbol map for filtering MT rows (matches Genebass's v29).
region_to_symbol = dict(zip(gene_coords['region'].to_list(), gene_coords['gene_name'].to_list()))
gene_coords.head()

## 5. Extract BETAs for the exact gene–phenotype pairs

Loop **per phenotype**: filter the MT to that one phenotype's column and to its associated genes' loci (via GENCODE intervals → `locus` index prune), then `.collect()` the entries and write a Polars parquet shard. Because each phenotype is paired only with its own genes, every collected row is a wanted `(gene, phenotype)` pair — no all-vs-all cross product.

**Input:** `mt`, `pheno_map`, `gene_coords`, `assoc`, `ensembl_to_symbol`. **Output:** one parquet shard per phenotype in `genebass_betas/shards_allvars/` with `id, region, BETA, SE, Pvalue, AF, phenotype, phenocode`. ~119 small Hail jobs; no full scan, no pandas.

In [ ]:
import os
from collections import defaultdict

# ---------------------------------------------------------------------------
# Extract BETAs for EXACTLY the 2,289 (gene, phenotype) pairs — not all-vs-all.
# Strategy: loop per phenotype. For each phenotype we know its associated genes,
# so we filter cols to that single phenotype and rows to those genes. Every entry
# collected is therefore a wanted (gene, phenotype) pair (no cross-product waste).
#
# Gene symbols come from GENCODE v29 (region_to_symbol) so they match the MT's
# `gene` field exactly (Genebass = VEP v95 = GENCODE v29). Each iteration:
#   filter_intervals (locus-index prune) -> filter_rows by v29 symbol
#   -> semi_join_cols to the one phenotype -> .collect() -> Polars -> parquet shard.
# Pandas is never touched; the Hail->Python boundary is .collect() (list of structs).
# ---------------------------------------------------------------------------

SHARD_DIR = 'PATH_TO_FILE'
os.makedirs(SHARD_DIR, exist_ok=True)

# Lookups
pheno_key = {r['phenotype']: r for _, r in pheno_map.iterrows()}      # phenotype -> col key fields
coords_by_region = {r['region']: r for r in gene_coords.to_dicts()}  # region -> v29 coords + symbol
matched_phenos = set(pheno_map['phenotype'])

# phenotype -> list of associated regions (only matched phenotypes with known v29 coords)
pairs_by_pheno = defaultdict(list)
for row in assoc.iter_rows(named=True):
    p, reg = row['phenotype'], row['region']
    if p in matched_phenos and reg in coords_by_region:
        pairs_by_pheno[p].append(reg)

PAD = 5_000  # bp padding so the gene-body interval safely covers all of the gene's variants
n = len(pairs_by_pheno)
for i, (pheno, regions) in enumerate(pairs_by_pheno.items()):
    regions = list(dict.fromkeys(regions))
    sym2region = {coords_by_region[r]['gene_name']: r for r in regions}   # v29 symbol -> Ensembl ID
    symbols = set(sym2region)

    intervals = []
    for r in regions:
        c = coords_by_region[r]
        lo = max(1, c['start'] - PAD)
        intervals.append(hl.locus_interval(c['contig'], lo, c['end'] + PAD + 1,
                                            reference_genome='GRCh38'))

    k = pheno_key[pheno]
    col_ht_p = hl.Table.parallelize(
        [hl.struct(trait_type=k['trait_type'], phenocode=k['phenocode'],
                   pheno_sex=k['pheno_sex'], coding=k['coding'], modifier=k['modifier'])],
        key=['trait_type', 'phenocode', 'pheno_sex', 'coding', 'modifier'],
    )

    m = hl.filter_intervals(mt, intervals)
    m = m.filter_rows(hl.literal(symbols).contains(m.gene))
    m = m.semi_join_cols(col_ht_p)
    ht = m.entries()
    ht = ht.select(
        id=ht.locus.contig + ':' + hl.str(ht.locus.position) + ':' + ht.alleles[0] + ':' + ht.alleles[1],
        gene_symbol=ht.gene,
        BETA=ht.BETA,
        SE=ht.SE,
        Pvalue=ht.Pvalue,
        AF=ht.AF,
        AC=ht.AC,
    )
    recs = ht.collect()
    if not recs:
        print(f"  [{i+1}/{n}] {pheno}: 0 variants ({len(regions)} genes)")
        continue

    sym2region_df = pl.DataFrame({'gene_symbol': list(sym2region), 'region': list(sym2region.values())})
    shard = (
        pl.DataFrame({
            'id':          [r.id for r in recs],
            'gene_symbol': [r.gene_symbol for r in recs],
            'BETA':        [r.BETA for r in recs],
            'SE':          [r.SE for r in recs],
            'Pvalue':      [r.Pvalue for r in recs],
            'AF':          [r.AF for r in recs],
            'AC':          [r.AC for r in recs],
        })
        .join(sym2region_df, on='gene_symbol', how='left')
        .with_columns([
            pl.lit(pheno).alias('phenotype'),
            pl.lit(k['phenocode']).alias('phenocode'),
        ])
        .drop('gene_symbol')
    )
    shard.write_parquet(f"{SHARD_DIR}/{pheno}.parquet")
    print(f"  [{i+1}/{n}] {pheno}: {shard.height:,} variant rows across {len(regions)} genes")

print(f"\nDone — shards written to {SHARD_DIR}")

## 6. Assemble & write the final parquet

**Input:** the per-phenotype shards + `pheno_map` (for `n_cases`). **Output:** the single final parquet `genebass_betas_127phenos_allvars.parquet`. Fully lazy in Polars (`scan_parquet → join → rename → sink_parquet`): the `BETA` column is renamed to **`mean_pheno_value`** (the UKBBGym APPV column name); final columns are `id, region, phenotype, phenocode, mean_pheno_value, SE, Pvalue, AF, n_cases`.

In [ ]:
# Assemble all per-phenotype shards into the final parquet — fully lazy Polars.
# scan_parquet -> join n_cases -> rename BETA to the UKBBGym APPV column name -> sink.

n_cases_lf = pl.from_pandas(pheno_map[['phenotype', 'n_cases']]).lazy()  # tiny 119-row metadata

(
    pl.scan_parquet(f"{SHARD_DIR}/*.parquet")
    .join(n_cases_lf, on='phenotype', how='left')
    .rename({'BETA': 'mean_pheno_value'})  # SAIGE single-variant BETA as APPV proxy
    .select(['id', 'region', 'phenotype', 'phenocode',
             'mean_pheno_value', 'SE', 'Pvalue', 'AF', 'n_cases'])
    .sink_parquet(OUT_PARQUET)
)

out = pl.read_parquet(OUT_PARQUET)
print(f"Final shape: {out.shape}")
print(f"Unique (region, phenotype) pairs: {out.select(['region', 'phenotype']).unique().height}")
print(f"Written to {OUT_PARQUET}")
out

---
# Quick check 
In the actual matrix table to confirm the same number of rows for a given (variant, gene, phenotype) pair:

In [ ]:
# Verify the output against the MT directly for a sample (gene, phenotype) pair.
MT_PATH = os.environ["GENEBASS_MT_PATH"]
OUT_PARQUET = 'PATH_TO_FILE'

# Use ENSG00000106105 — one of the 7 genes renamed since 2018 (v29 'GARS' / v39 'GARS1').
# Under v39 symbols this gene matched nothing; under v29 it should now have variants.
# CHECK_REGION = 'ENSG00000106105'
# CHECK_REGION = 'ENSG00000162551' #ALPL
# CHECK_PHENO  = 'alkaline_phosphatase_int'

CHECK_REGION = 'ENSG00000130164' #LDLR
CHECK_PHENO  = 'ldl_direct_int'

In [ ]:
# (a) count in our output parquet
out = pl.read_parquet(OUT_PARQUET).filter((pl.col('region') == CHECK_REGION) & (pl.col('phenotype') == CHECK_PHENO))
out

In [ ]:
CHECK_REGION_NAME = ensembl_to_symbol[CHECK_REGION]
CHECK_PHENOCODE  = out.select('phenocode').unique().to_series().to_list()[0]
CHECK_PHENOCODE

In [ ]:
mt = hl.read_matrix_table(MT_PATH)
mt.describe()

In [ ]:
# pl.read_parquet(OUT_PARQUET).filter(pl.col('id') == 'REDACTED_VARIANT_ID')
# out.filter(pl.col('id') == 'REDACTED_VARIANT_ID')
out.filter(pl.col('id') == 'REDACTED_VARIANT_ID')

In [ ]:
# (b) count straight from the MT (independent of the extraction pipeline)
c = coords_by_region[CHECK_REGION]
k = pheno_key[CHECK_PHENO]
iv = [hl.locus_interval(c['contig'], max(1, c['start'] - 5000), c['end'] + 5001,
                        reference_genome='GRCh38')]
col_ht = hl.Table.parallelize(
    [hl.struct(trait_type=k['trait_type'], phenocode=k['phenocode'],
               pheno_sex=k['pheno_sex'], coding=k['coding'], modifier=k['modifier'])],
    key=['trait_type', 'phenocode', 'pheno_sex', 'coding', 'modifier'])
m = hl.filter_intervals(mt, iv)
m = m.filter_rows(m.gene == c['gene_name'])
m = m.semi_join_cols(col_ht)
n_mt = m.entries().count()

print(f"{CHECK_REGION} ({c['gene_name']}) x {CHECK_PHENO}")
print(f"  rows in output parquet : {out.height}")
print(f"  entries in MT directly : {n_mt}")
print("  MATCH ✓" if out.height == n_mt and n_mt > 0 else "  MISMATCH ✗")